# 02 - 特征工程实验

本笔记本用于探索和优化特征工程流程，找出对水印检测最有效的特征组合。

## 1. 导入必要的库

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.feature_extractor import FeatureExtractor

%matplotlib inline

## 2. 加载数据

In [ ]:
# 加载数据
loader = DataLoader(time_column='timestamp')
df = loader.load('../data/sample/sample_data.csv')

print(f"数据形状: {df.shape}")
df.head()

## 3. 特征提取实验

In [ ]:
# 初始化特征提取器
extractor = FeatureExtractor(
    window_size=5,
    step_size=1,
    feature_groups=['statistical', 'time_domain', 'frequency_domain', 'complexity', 'watermark_specific']
)

# 提取特征
features_df = extractor.extract(df, target_column='close')

print(f"特征数据形状: {features_df.shape}")
print(f"\n特征列表:")
feature_cols = [c for c in features_df.columns if c not in ['window_start', 'window_end']]
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

## 4. 特征分析

In [ ]:
# 特征统计
features_df[feature_cols].describe()

In [ ]:
# 特征分布可视化
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
axes = axes.flatten()

for idx, col in enumerate(feature_cols[:25]):
    axes[idx].hist(features_df[col].dropna(), bins=20, alpha=0.7, edgecolor='black')
    axes[idx].set_title(col)
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')

# 隐藏多余的子图
for idx in range(len(feature_cols[:25]), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 5. 特征相关性分析

In [ ]:
# 计算特征相关性
corr_matrix = features_df[feature_cols].corr()

# 绘制相关性热力图
plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 找出高度相关的特征对
high_corr_pairs = []
threshold = 0.9

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i], 
                corr_matrix.columns[j], 
                corr_matrix.iloc[i, j]
            ))

print(f"高度相关的特征对 (|correlation| > {threshold}):")
for feat1, feat2, corr in high_corr_pairs:
    print(f"  {feat1} <-> {feat2}: {corr:.3f}")

## 6. 特征重要性分析

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

# 准备数据
X = features_df[feature_cols].values

# 获取标签（从原始数据中）
y = []
for _, row in features_df.iterrows():
    window_data = df.loc[row['window_start']:row['window_end']]
    if 'watermark_label' in window_data.columns:
        label = window_data['watermark_label'].mode()[0]
    else:
        label = 0
    y.append(label)
y = np.array(y)

print(f"样本数: {len(X)}, 正样本数: {sum(y)}, 负样本数: {len(y) - sum(y)}")

In [ ]:
# 使用随机森林计算特征重要性
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# 获取特征重要性
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

# 可视化
plt.figure(figsize=(12, 8))
plt.barh(range(len(importance_df)), importance_df['importance'])
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 重要特征:")
print(importance_df.head(10))

In [ ]:
# 互信息分析
mi_scores = mutual_info_classif(X, y, random_state=42)

mi_df = pd.DataFrame({
    'feature': feature_cols,
    'mutual_info': mi_scores
}).sort_values('mutual_info', ascending=False)

# 可视化
plt.figure(figsize=(12, 8))
plt.barh(range(len(mi_df)), mi_df['mutual_info'])
plt.yticks(range(len(mi_df)), mi_df['feature'])
plt.xlabel('Mutual Information')
plt.title('Feature Mutual Information with Target')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 互信息特征:")
print(mi_df.head(10))

## 7. 不同特征组的效果对比

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# 定义要测试的特征组
feature_groups = {
    'statistical': ['statistical'],
    'time_domain': ['time_domain'],
    'frequency_domain': ['frequency_domain'],
    'complexity': ['complexity'],
    'watermark_specific': ['watermark_specific'],
    'statistical+time': ['statistical', 'time_domain'],
    'statistical+freq': ['statistical', 'frequency_domain'],
    'all': ['statistical', 'time_domain', 'frequency_domain', 'complexity', 'watermark_specific']
}

results = []

for name, groups in feature_groups.items():
    # 提取特定特征组的特征
    extractor_test = FeatureExtractor(
        window_size=5,
        step_size=1,
        feature_groups=groups
    )
    
    features_test = extractor_test.extract(df, target_column='close')
    feature_cols_test = [c for c in features_test.columns if c not in ['window_start', 'window_end']]
    
    if len(feature_cols_test) == 0:
        continue
    
    X_test = features_test[feature_cols_test].values
    
    # 交叉验证
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    scores = cross_val_score(clf, X_test, y, cv=3, scoring='f1')
    
    results.append({
        'feature_group': name,
        'num_features': len(feature_cols_test),
        'f1_mean': scores.mean(),
        'f1_std': scores.std()
    })
    
    print(f"{name:20s}: {len(feature_cols_test):2d} features, F1 = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

results_df = pd.DataFrame(results)

In [ ]:
# 可视化对比结果
plt.figure(figsize=(12, 6))
plt.bar(results_df['feature_group'], results_df['f1_mean'], 
        yerr=results_df['f1_std'], capsize=5, alpha=0.7)
plt.xlabel('Feature Group')
plt.ylabel('F1 Score')
plt.title('Feature Group Performance Comparison')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 窗口大小参数调优

In [ ]:
# 测试不同的窗口大小
window_sizes = [3, 5, 10, 20, 30]
window_results = []

for window_size in window_sizes:
    extractor_window = FeatureExtractor(
        window_size=window_size,
        step_size=max(1, window_size // 5),
        feature_groups=['statistical', 'time_domain', 'watermark_specific']
    )
    
    features_window = extractor_window.extract(df, target_column='close')
    feature_cols_window = [c for c in features_window.columns if c not in ['window_start', 'window_end']]
    
    X_window = features_window[feature_cols_window].values
    
    # 获取标签
    y_window = []
    for _, row in features_window.iterrows():
        window_data = df.loc[row['window_start']:row['window_end']]
        if 'watermark_label' in window_data.columns:
            label = window_data['watermark_label'].mode()[0]
        else:
            label = 0
        y_window.append(label)
    y_window = np.array(y_window)
    
    # 交叉验证
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    scores = cross_val_score(clf, X_window, y_window, cv=3, scoring='f1')
    
    window_results.append({
        'window_size': window_size,
        'num_samples': len(X_window),
        'f1_mean': scores.mean(),
        'f1_std': scores.std()
    })
    
    print(f"Window size {window_size:2d}: {len(X_window):3d} samples, F1 = {scores.mean():.4f}")

window_results_df = pd.DataFrame(window_results)

In [ ]:
# 可视化窗口大小对比
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1分数
axes[0].plot(window_results_df['window_size'], window_results_df['f1_mean'], 'o-', linewidth=2, markersize=8)
axes[0].fill_between(window_results_df['window_size'], 
                     window_results_df['f1_mean'] - window_results_df['f1_std'],
                     window_results_df['f1_mean'] + window_results_df['f1_std'],
                     alpha=0.3)
axes[0].set_xlabel('Window Size')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Window Size vs F1 Score')
axes[0].grid(True, alpha=0.3)

# 样本数量
axes[1].bar(window_results_df['window_size'], window_results_df['num_samples'], alpha=0.7)
axes[1].set_xlabel('Window Size')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Window Size vs Number of Samples')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 总结与建议

基于以上实验，我们可以得出以下结论：

1. **最重要的特征**: 根据特征重要性分析，列出最重要的前几个特征
2. **最佳特征组合**: 根据对比实验，确定最有效的特征组合
3. **最优窗口大小**: 根据窗口大小实验，选择合适的窗口大小
4. **特征筛选建议**: 基于相关性分析，建议去除高度冗余的特征

这些结论将用于优化配置文件中的特征工程参数。